# Statistika: A/B test
## Gec çatdırılma müştəri rəyini azaldırmı?

Analiz edirik gec çatdırılan sifarişlərin rəy balının vaxtında çatdırılanlardan aşağı olub-olmadığını.

Sifarişləri iki qrupa bölürük və qrupların orta rəy balını müqayisə edirik. Sonra statistik testlə fərqin təsadüfi olub-olmadığını yoxlayırıq.

| Qrup | Kimlərdir |
|---|---|
| A | Vaxtında çatdırılan sifarişlər |
| B | Gec çatdırılan sifarişlər |

Sifariş gözlənilən tarixdən sonra müştəriyə çatıbsa, gec sayılır.

silver.orders və silver.reviews cədvəlləri. Yalnız statusu delivered olan və çatdırılma tarixi dolu olan sifarişləri götürürük.


## Məzmun
1. Addım 1: məlumatı oxumaq
2. Addım 2: gec və vaxtında qruplarını qurmaq
3. Addım 3: orta rəy ballarını müqayisə etmək
4. Addım 4: statistik test
5. Yekun nəticə

In [2]:
import sys
from pathlib import Path
import pandas as pd

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists())
sys.path.append(str(_root / "notebooks"))

from db import get_engine
engine = get_engine()

In [3]:
print("orders sütunları:", pd.read_sql("SELECT TOP 1 * FROM silver.orders", engine).columns.tolist())
print("reviews sütunları:", pd.read_sql("SELECT TOP 1 * FROM silver.reviews", engine).columns.tolist())

orders sütunları: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
reviews sütunları: ['review_id', 'order_id', 'review_score', 'comment_title', 'comment_message', 'creation_date', 'answer_timestamp']


## Step1: məlumatı oxumaq
silver.orders və silver.reviews cədvəllərini order_id üzrə birləşdiririk. Yalnız statusu 
delivered olan və çatdırılma tarixi dolu olan sifarişləri götürürük, çünki çatmayan sifarişin vaxtında gəlib-gəlmədiyini deyə bilmərik.


In [4]:
sql = """
SELECT o.order_id,
       o.order_delivered_customer_date AS delivered,
       o.order_estimated_delivery_date AS estimated,
       r.review_score
FROM silver.orders o
JOIN silver.reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
"""
df = pd.read_sql(sql, engine)

In [5]:
df["review_score"] = pd.to_numeric(df["review_score"])

In [6]:
# 1.3 Neçə sətir var?
len(df)

96353

birləşdirmədən sonra 96,353 sətir var və 529-u təkrarlanan order_id-dir. Bəzi sifarişlərə bir neçə rəy yazılıb. Hər sifarişi bir dəfə saymaq üçün onların rəy ballarının ortalamasını götürəcəyik. Təkrarsız sifariş sayı 95,824 olacaq. Rəyi olmayan çatdırılmış sifarişlər analizə düşmür, yəni nəticə yalnız rəy yazan müştərilər haqqındadır.

In [7]:

df["order_id"].duplicated().sum()

np.int64(529)

In [8]:
df.head()

,order_id,delivered,estimated,review_score
0,998b1beee068744ac833dd8b31944623,2018-06-05 17:02:57,2018-05-22,1
1,75cc0defdf90cda65066ce0ac171e2ab,2017-10-14 15:43:39,2017-11-06,5
2,df940e48580d0f2c2283d0782c7f99e3,2017-08-25 23:29:54,2017-09-13,5
3,dc6a29077e547f5fe4377f251fd1c5d0,2017-06-02 15:08:57,2017-06-13,5
4,c7068b2d09a51ca53dd0a06c2526be34,2018-05-02 18:12:32,2018-05-17,4


In [9]:
dup = df[df["order_id"].duplicated(keep=False)].sort_values("order_id")
dup.head(6)

,order_id,delivered,estimated,review_score
64774,0035246a40f520710769010f752e7507,2017-08-28 19:14:03,2017-09-06,5
62548,0035246a40f520710769010f752e7507,2017-08-28 19:14:03,2017-09-06,5
51220,013056cfe49763c6f66bda03396c5ee3,2018-03-03 12:09:00,2018-03-07,5
10818,013056cfe49763c6f66bda03396c5ee3,2018-03-03 12:09:00,2018-03-07,4
92722,0176a6846bcb3b0d3aa3116a9a768597,2017-12-29 21:22:35,2018-01-15,5
84636,0176a6846bcb3b0d3aa3116a9a768597,2017-12-29 21:22:35,2018-01-15,5


In [10]:
df = df.groupby(["order_id", "delivered", "estimated"], as_index=False)["review_score"].mean()

In [11]:
len(df), df["order_id"].duplicated().sum()

(95824, np.int64(0))

 96,353 sətir oxuduq. Onlardan 529-u təkrarlanan order_id idi, çünki bəzi sifarişlərə bir neçə rəy yazılıb eyni sifarişdə 5 və 4. Hər sifarişi bir dəfə saymaq üçün rəy ballarının ortalamasını götürdük. Nəticədə 95,824 sifariş qaldı və təkrar yoxdur.
 
  Qeyd olraq analiz yalnız rəy yazan müştərilər haqqındadır, rəyi olmayan çatdırılmış sifarişlər bura düşmür.

## Step2  gec və vaxtında qruplarını qurmaq
Hər sifariş üçün baxırıq müştəriyə çatdırılma günü gözlənilən gündən sonradırmı? Çatdırılma tarixində saat var, gözlənilən tarixdə isə yoxdur. Gözlənilən gün çatdırılanların səhvən gec sayılmaması üçün yalnız günləri müqayisə edirik.

In [12]:
df["delivered"] = pd.to_datetime(df["delivered"])
df["estimated"] = pd.to_datetime(df["estimated"])

In [13]:
df["is_late"] = df["delivered"].dt.normalize() > df["estimated"]

In [14]:
df["is_late"].value_counts()

is_late
False    89443
True      6381
Name: count, dtype: int64

In [15]:
(df["is_late"].value_counts(normalize=True) * 100).round(1)

is_late
False    93.3
True      6.7
Name: proportion, dtype: float64

Nəticə olaraq 95,824 sifarişin 89,443-ü (93.3%) vaxtında, 6,381-i (6.7%) gec çatdırılıb. Qruplar bərabər ölçüdə deyil: gec çatdırılan sifarişlər azlıq təşkil edir.

## Step3 orta rəy ballarını müqayisə etmək
İki qrupun orta rəy balına baxırıq. Sonra fərqi hesablayırıq və hər qrupda aşağı bal 2 və ya daha az verənlərin faizini müqayisə edirik. 

In [16]:
df.groupby("is_late")["review_score"].mean().round(2)

is_late
False    4.29
True     2.27
Name: review_score, dtype: float64

In [17]:
ortalama = df.groupby("is_late")["review_score"].mean()
round(ortalama[False] - ortalama[True], 2)

np.float64(2.02)

In [18]:
(df["review_score"] <= 2).groupby(df["is_late"]).mean().mul(100).round(1)

is_late
False     9.2
True     62.4
Name: review_score, dtype: float64

Nəticə olaraq vaxtında çatdırılan sifarişlərin orta rəy balı 4.29, gec çatdırılanlarınki isə 2.27-dir. Fərq 2.02 baldır. Aşağı bal  verənlərin faizi vaxtında qrupda 9.2%, gec qrupda 62.4%-dir. Deməli gec çatdırılan sifarişlərin rəyi xeyli aşağıdır. Bu fərqin təsadüfi olub-olmadığını yoxlayacıq

### Hipotezlər və qərar qaydası
- **H0 hipotez:** gec və vaxtında çatdırılan sifarişlərin rəy balları arasında fərq yoxdur.
- **H1 (alternativ hipotez):** rəy balları arasında fərq var.
- **Əhəmiyyətlilik səviyyəsi:** 0.05. p-dəyər 0.05-dən kiçikdirsə, H0-ı rədd edirik, yəni fərq təsadüfi sayılmır.

## Step4 statistik test
Orta rəy ballarının fərqli olduğunu gördük 4.29 və 2.27. İndi bu fərqin təsadüfi olub-olmadığını yoxlayırıq. Mann-Whitney testindən istifadə edirik, çünki rəy balı 1-dən 5-ə qədər tam rəqəmdir və zəng şəkilli paylanmır. Test p-dəyər verir: 0.05-dən kiçikdirsə, fərq təsadüfi sayılmır.


In [19]:
vaxtinda = df[df["is_late"] == False]["review_score"]
gec = df[df["is_late"] == True]["review_score"]
len(vaxtinda), len(gec)

(89443, 6381)

In [20]:
from scipy.stats import mannwhitneyu
netice = mannwhitneyu(vaxtinda, gec)
netice

MannwhitneyuResult(statistic=np.float64(467426830.5), pvalue=np.float64(0.0))

In [21]:
print("p-dəyər:", netice.pvalue)
print("Fərq təsadüfi deyil (p < 0.05):", netice.pvalue < 0.05)

p-dəyər: 0.0
Fərq təsadüfi deyil (p < 0.05): True


In [22]:
# Fərqin böyüklüyü: təsadüfi seçilmiş vaxtında sifarişin balı gec sifarişdən nə qədər tez yüksək olur?
netice.statistic / (len(vaxtinda) * len(gec))

np.float64(0.8189898801211137)

### Yekun nəticə: 

- 95,824 sifarişin 89,443-ü (93.3%) vaxtında, 6,381-i (6.7%) gec çatdırılıb.
- Orta rəy balı: vaxtında 4.29, gec 2.27 (fərq 2.02 bal).
- Aşağı bal (2 və ya aşağı) verənlər: vaxtında 9.2%, gec 62.4%.
- Mann-Whitney testi: p < 0.001, fərq təsadüfi deyil.

gec çatdırılan sifarişlərin rəy balı vaxtında çatdırılanlardan xeyli aşağıdır və bu fərq təsadüfi deyil. Biznes üçün mənası: çatdırılmanı vaxtında təmin etmək müştəri məmnuniyyəti üçün vacibdir.

# B2. Yağışlı günlərdə sifariş sayı adi günlərdən niyə çoxdur ?

São Paulo-da yağışlı günlərdə gündəlik sifariş sayı yağışsız günlərdən fərqlənirmi?
SQL report-da (B4) yağışlı günlərdə orta sifariş sayı 91.83, yağışsız günlərdə 81.77 çıxdı. Burada bu fərqin təsadüfi olub-olmadığını yoxlayırıq və səbəbini axtarırıq.

## Hipotezlər
- **H0:** yağışlı və yağışsız günlərin sifariş sayında fərq yoxdur
- **H1:** fərq var
- Əhəmiyyətlilik səviyyəsi: α = 0.05

## Test
Mann-Whitney. Gündəlik sifariş sayı normal paylanmaya bilər, ona görə t-test əvəzinə bunu seçirik.

## Addımlar
1. Məlumatı hazırlamaq: hər gün üçün sifariş sayı və yağış qrupu
2. İlk baxış: hər qrupda gün sayı, orta və median sifariş
3. Test: Mann-Whitney, p-value və təsir ölçüsü
4. Yoxlama: fərq yağışdan gəlir, yoxsa il və mövsümdən?
5. Nəticə və məhdudiyyətlər

## Step1.
Hər gün üçün sifariş sayını və hava qrupunu bir cədvəldə toplayiriq Yağış > 0 olan günlər Rainy, qalanları Dry sayılır.

Cədvəldə 1096 gün var, hər birində tarix, gündəlik sifariş sayı və hava qrupu göstərilib.

In [23]:
df = pd.read_sql("""
    SELECT
        sales_date,
        total_orders,
        CASE WHEN sp_precipitation > 0 THEN 'Rainy' ELSE 'Dry' END AS weather_type
    FROM gold.mart_daily_sales_weather
""", engine)

print(df.shape)
df.head()

(1096, 3)


,sales_date,total_orders,weather_type
0,2017-04-11,73,Dry
1,2016-10-07,38,Rainy
2,2017-10-11,151,Rainy
3,2017-06-30,93,Dry
4,2017-12-16,129,Dry


Hər hava qrupunda gün sayını, orta və median sifariş sayını müqayisə etmək. Median əlavə edilib, çünki orta çox yüksək günlərdən təsirlənə bilər.

Yağışlı günlərdə orta sifariş sayı daha yüksəkdir , amma median daha aşağıdır (59.5 qarşı 69.0). Bu, yağışlı qrupda bir neçə çox yüksək sifarişli gün olduğunu və ortanı yuxarı çəkdiyini göstərir. Deməli, yağışlı günlərdə "daha çox sifariş" nəticəsi birmənalı deyil.

In [24]:
df.groupby("weather_type")["total_orders"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
weather_type,,,
Dry,414,81.77,69.0
Rainy,682,91.83,59.5


## 3. Mann-Whitney testi

 Yağışlı və yağışsız günlərin gündəlik sifariş sayındakı fərqin təsadüfi olub-olmadığını yoxlamaq.

 p-value 0.30 çıxdı α = 0.05-dən böyükdür, ona görə H0-ı rədd edə bilmirik. Təsir ölçüsü 0.518 təşkil edir, yəni 0.5-ə çox yaxındır və iki qrup arasında praktiki fərq yoxdur. Deməli, B4-dəki orta fərq (91.83 qarşı 81.77) statistik cəhətdən əhəmiyyətli deyil və yağışın sifarişi artırdığını deməyə əsas yoxdur.

In [25]:
from scipy.stats import mannwhitneyu

rainy = df[df["weather_type"] == "Rainy"]["total_orders"]
dry = df[df["weather_type"] == "Dry"]["total_orders"]

u, p = mannwhitneyu(rainy, dry, alternative="two-sided")

print("p-value:", p)
print("Effect size:", round(u / (len(rainy) * len(dry)), 3))

p-value: 0.3010792860872722
Effect size: 0.518


## 4.
Orta sifariş fərqinin yağışdan, yoxsa zamandan gəldiyini yoxlayiriq. Hər il üçün gün sayı, yağışlı günlərin payı və orta sifariş hesablanıb.

Satış illər üzrə kəskin artıb. Eyni zamanda yağışlı günlərin payı da artıb. Deməli, yağışlı günlər çox sifarişli illərə daha çox düşür və qruplar arasındakı orta fərq yağışdan yox, zaman trendindən qaynaqlana bilər.

In [26]:
df["year"] = pd.to_datetime(df["sales_date"]).dt.year

df.groupby("year").agg(
    days=("total_orders", "count"),
    rainy_pct=("weather_type", lambda x: round((x == "Rainy").mean() * 100, 2)),
    avg_orders=("total_orders", "mean"),
).round(2)

,days,rainy_pct,avg_orders
year,,,
2016,366,54.92,0.73
2017,365,62.47,118.98
2018,365,69.32,144.61


## 5.
 2016-da məlumat demək olar ki, yoxdur, bu da müqayisəni pozur. Yalnız 2017-2018 üçün qrupları yenidən müqayisə edirik.

2016 çıxarıldıqdan sonra yağışsız günlərdə sifariş daha yüksəkdir orta 135.45 qarşı 129.91, median 134 qarşı 122. Yəni ilkin "yağışlı günlərdə daha çox sifariş" nəticəsi 2016-nın təsirindən yaranmışdı.

In [27]:
df2 = df[df["year"] >= 2017]

df2.groupby("weather_type")["total_orders"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
weather_type,,,
Dry,249,135.45,134.0
Rainy,481,129.91,122.0


## 6. Mann-Whitney testi (2017-2018)

2016 çıxarıldıqdan sonra yağışlı və yağışsız günlərin sifariş sayındakı fərqi təkrar yoxlayiriq.

p-value 0.17 çıxdı α = 0.05-dən böyükdür, ona görə H0-ı rədd edə bilmirik. Təsir ölçüsü 0.469 təşkil edir yəni 0.5-ə yaxındır və yağışsız günlər cüzi üstündür amma bu fərq statistik əhəmiyyətli deyil. Deməli yağışın gündəlik sifariş sayına təsirinə dair sübut yoxdur.

In [28]:
rainy = df2[df2["weather_type"] == "Rainy"]["total_orders"]
dry = df2[df2["weather_type"] == "Dry"]["total_orders"]

u, p = mannwhitneyu(rainy, dry, alternative="two-sided")

print("p-value:", p)
print("Effect size:", round(u / (len(rainy) * len(dry)), 3))

p-value: 0.1713809260923328
Effect size: 0.469


## 7. Yekun nəticə

- B4-dəki "yağışlı günlərdə daha çox sifariş" nəticəsi yağışdan gəlmirdi. Yağışlı günlər çox sifarişli illərə daha çox düşdüyü üçün belə görünürdü.
- Gündəlik sifariş sayına yağışın statistik əhəmiyyətli təsiri yoxdur.

**Biznes mənası:** Sifariş planlaşdırarkən yağış faktoru əsas götürülməməlidir. Əvvəl SQL report-da gecikmənin yağışla artdığını gördük (4.24% qarşı 8.14%), yəni yağışın təsiri satışda yox, **çatdırılmada** nəzərə çarpır.